# **Training Model**

In [3]:
# General Libraries
import os
import pandas as pd
import numpy as np

# Metrics
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# Databricks Env
import pathlib
import pickle
from dotenv import load_dotenv

# Feature Engineering
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Optimization
import math
import optuna
from optuna.samplers import TPESampler

# MLFlow
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
from mlflow import MlflowClient

# Modeling
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier

# Evaluation Metrics
from sklearn.metrics import accuracy_score, precision_score, f1_score, recall_score

from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
# safe_databricks_setup.py
from dotenv import load_dotenv
import os
import mlflow

In [4]:
import mlflow
mlflow.sklearn.autolog()

In [5]:
import mlflow

# Force-end any active run
while mlflow.active_run() is not None:
    mlflow.end_run()

In [6]:
# Load .env and Log in to Databricks

# Cargar las variables del archivo .env
load_dotenv(override=True)  # Carga las variables del archivo .env
EXPERIMENT_NAME = "/Users/aclarapao@gmail.com/proyecto_final_precios_4" 

mlflow.set_tracking_uri("databricks")
experiment = mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)

2025/12/02 14:28:23 INFO mlflow.tracking.fluent: Experiment with name '/Users/aclarapao@gmail.com/proyecto_final_precios_4' does not exist. Creating a new experiment.


In [ ]:
import os
import pickle
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import mlflow

# ---- load
df = pd.read_csv('../data/processed/df_clean.csv')

# split
y = df["price"]
X = df.drop(columns=["price"])

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=42, shuffle=False)

def preprocessor(X_train, X_test, X_val=None, save_data=False, save_artifacts=True):
    # Make copies so we don't mutate outside variables
    X_train = X_train.copy()
    X_test  = X_test.copy()
    X_val   = X_val.copy() if X_val is not None else None

    # 1) Impute missing values
    # Fill pets_allowed with 0 (assumption: NaN => no pets allowed)
    if 'pets_allowed' in X_train.columns:
        X_train['pets_allowed'] = X_train['pets_allowed'].fillna(0)
        X_test['pets_allowed']  = X_test['pets_allowed'].fillna(0)
        if X_val is not None:
            X_val['pets_allowed'] = X_val['pets_allowed'].fillna(0)

    # For other numeric columns use train median
    numeric_cols = ['bathrooms', 'bedrooms', 'square_feet', 'latitude', 'longitude', 'amenities_count']
    for col in numeric_cols:
        if col in X_train.columns:
            med = X_train[col].median()
            X_train[col] = X_train[col].fillna(med)
            X_test[col]  = X_test[col].fillna(med)
            if X_val is not None:
                X_val[col] = X_val[col].fillna(med)

    # 2) One-Hot encode cityname and state together
    cat_cols = [c for c in ["category", "has_photo", "pets_allowed", "cityname", "state"] if c in X_train.columns]
    encoder = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
    if len(cat_cols) > 0:
        encoder.fit(X_train[cat_cols])

        X_train_cat = encoder.transform(X_train[cat_cols])
        X_test_cat  = encoder.transform(X_test[cat_cols])
        X_val_cat   = encoder.transform(X_val[cat_cols]) if X_val is not None else None

        cat_feature_names = encoder.get_feature_names_out(cat_cols)

        X_train_cat_df = pd.DataFrame(X_train_cat, columns=cat_feature_names, index=X_train.index)
        X_test_cat_df  = pd.DataFrame(X_test_cat,  columns=cat_feature_names, index=X_test.index)
        X_val_cat_df   = pd.DataFrame(X_val_cat,   columns=cat_feature_names, index=X_val.index) if X_val is not None else None

        # drop original cat cols and concat encoded
        X_train = X_train.drop(columns=cat_cols)
        X_test  = X_test.drop(columns=cat_cols)
        X_val   = X_val.drop(columns=cat_cols) if X_val is not None else None

        X_train_final = pd.concat([X_train, X_train_cat_df], axis=1)
        X_test_final  = pd.concat([X_test,  X_test_cat_df],  axis=1)
        X_val_final   = pd.concat([X_val,   X_val_cat_df],   axis=1) if X_val is not None else None
    else:
        # no categorical cols found
        X_train_final = X_train
        X_test_final  = X_test
        X_val_final   = X_val

    # 3) Scale (fit on train only)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_final)
    X_test_scaled  = scaler.transform(X_test_final)
    X_val_scaled   = scaler.transform(X_val_final) if X_val is not None else None

    feature_cols = list(X_train_final.columns)
    # 4) save scaled dataframes to csv
    if save_data:
        X_train_df = pd.DataFrame(X_train_scaled, columns=feature_cols, index=X_train_final.index)
        X_test_df  = pd.DataFrame(X_test_scaled,  columns=feature_cols, index=X_test_final.index)
        X_val_df   = pd.DataFrame(X_val_scaled,   columns=feature_cols, index=X_val_final.index) if X_val is not None else None

        X_train_df.to_csv('../data/processed/X_train.csv', index=False)
        X_test_df.to_csv('../data/processed/X_test.csv', index=False)
        if X_val_df is not None:
            X_val_df.to_csv('../data/processed/X_val.csv', index=False)

    # 5) Save artifacts
    if save_artifacts:
        os.makedirs("../artifacts/preprocessor", exist_ok=True)
        with open('../artifacts/preprocessor/encoder.pkl', 'wb') as f_out:
            pickle.dump(encoder, f_out)
        with open('../artifacts/preprocessor/scaler.pkl', 'wb') as f_out:
            pickle.dump(scaler, f_out)
        with open('../artifacts/preprocessor/feature_columns.pkl', 'wb') as f_out:
            pickle.dump(feature_cols, f_out)


    # Return scaled arrays + artifacts + feature names so user can reconstruct dfs
    return X_train_scaled, X_test_scaled, X_val_scaled, encoder, scaler, list(X_train_final.columns)

# ---- call the preprocessor (note updated return unpacking)
X_train_scaled, X_test_scaled, X_val_scaled, encoder, scaler, feature_cols = preprocessor(
    X_train, X_test, X_val, save_data=True, save_artifacts=True
)



# reconstruct DataFrames 
X_train_df = pd.DataFrame(X_train_scaled, columns=feature_cols, index=X_train.index)
X_test_df  = pd.DataFrame(X_test_scaled,  columns=feature_cols, index=X_test.index)
X_val_df   = pd.DataFrame(X_val_scaled,   columns=feature_cols, index=X_val.index)

# --- Quick checks 
print("Shapes after scaling:")
print("X_train_df:", X_train_df.shape, " y_train:", y_train.shape)
print("X_val_df:  ", X_val_df.shape,   " y_val:", y_val.shape)
print("X_test_df: ", X_test_df.shape,  " y_test:", y_test.shape)

print("\nNaNs after preprocessing:")
print("X_train_df NaNs:", X_train_df.isna().sum().sum())
print("X_val_df NaNs:  ", X_val_df.isna().sum().sum())
print("X_test_df NaNs: ", X_test_df.isna().sum().sum())

# show small heads
display(X_train_df.head())
display(y_train.head())
display(X_val_df.head())
display(y_val.head())
display(X_test_df.head())
display(y_test.head())

c:\Users\diana\anaconda3\envs\lab_modelado\Lib\site-packages\sklearn\preprocessing\_encoders.py:242: UserWarning: Found unknown categories in columns [3, 4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
c:\Users\diana\anaconda3\envs\lab_modelado\Lib\site-packages\sklearn\preprocessing\_encoders.py:242: UserWarning: Found unknown categories in columns [0, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


Shapes after scaling:
X_train_df: (5321, 1103)  y_train: (5321,)
X_val_df:   (1774, 1103)  y_val: (1774,)
X_test_df:  (1774, 1103)  y_test: (1774,)

NaNs after preprocessing:
X_train_df NaNs: 0
X_val_df NaNs:   0
X_test_df NaNs:  0


,bathrooms,bedrooms,square_feet,latitude,longitude,amenities_count,category_housing/rent/home,has_photo_Thumbnail,has_photo_Yes,cityname_2,...,state_40,state_41,state_42,state_43,state_44,state_45,state_46,state_47,state_48,state_49
0,-0.123808,-2.219288,-3.611774,0.244347,1.211648,-0.969858,-0.019391,0.311267,-0.282009,-0.01371,...,-0.049489,-0.075299,-0.099343,-0.562519,-0.087031,-0.147300,-0.033599,-0.235679,-0.169732,-0.01371
1,-0.123808,-0.376174,-3.577679,0.082397,0.509396,-0.969858,-0.019391,0.311267,-0.282009,-0.01371,...,-0.049489,-0.075299,-0.099343,-0.562519,-0.087031,-0.147300,-0.033599,-0.235679,-0.169732,-0.01371
2,-0.123808,-2.219288,-3.570860,0.241808,1.205366,-0.969858,-0.019391,0.311267,-0.282009,-0.01371,...,-0.049489,-0.075299,-0.099343,-0.562519,-0.087031,6.788851,-0.033599,-0.235679,-0.169732,-0.01371
3,-0.123808,-2.219288,-3.509490,1.748699,-1.770845,-0.969858,-0.019391,0.311267,-0.282009,-0.01371,...,-0.049489,-0.075299,-0.099343,-0.562519,-0.087031,-0.147300,-0.033599,4.243062,-0.169732,-0.01371
4,-0.123808,-2.219288,-3.448120,0.238837,1.203794,-0.969858,-0.019391,0.311267,-0.282009,-0.01371,...,-0.049489,-0.075299,-0.099343,-0.562519,-0.087031,6.788851,-0.033599,-0.235679,-0.169732,-0.01371


0     790
1     425
2    1390
3     925
4     880
Name: price, dtype: int64

,bathrooms,bedrooms,square_feet,latitude,longitude,amenities_count,category_housing/rent/home,has_photo_Thumbnail,has_photo_Yes,cityname_2,...,state_40,state_41,state_42,state_43,state_44,state_45,state_46,state_47,state_48,state_49
5321,-0.123808,1.466940,1.495587,-0.587388,-1.505474,0.157714,-0.019391,0.311267,-0.282009,-0.01371,...,-0.049489,-0.075299,-0.099343,-0.562519,-0.087031,-0.1473,-0.033599,-0.235679,-0.169732,-0.01371
5322,-0.123808,1.466940,1.495587,-0.596490,-1.505737,-0.969858,-0.019391,0.311267,-0.282009,-0.01371,...,-0.049489,-0.075299,-0.099343,-0.562519,-0.087031,-0.1473,-0.033599,-0.235679,-0.169732,-0.01371
5323,-0.123808,-0.376174,1.495587,-0.596490,-1.505737,-0.969858,-0.019391,0.311267,-0.282009,-0.01371,...,-0.049489,-0.075299,-0.099343,-0.562519,-0.087031,-0.1473,-0.033599,-0.235679,-0.169732,-0.01371
5324,-0.123808,-0.376174,1.495587,0.484206,1.408385,1.849072,-0.019391,0.311267,-0.282009,-0.01371,...,-0.049489,-0.075299,-0.099343,-0.562519,-0.087031,-0.1473,-0.033599,-0.235679,-0.169732,-0.01371
5325,-0.123808,-0.376174,1.495587,0.850955,1.594426,-0.687965,-0.019391,-3.212674,3.545981,-0.01371,...,-0.049489,-0.075299,-0.099343,-0.562519,-0.087031,-0.1473,-0.033599,-0.235679,-0.169732,-0.01371


5321    2700
5322    2695
5323    2570
5324    2515
5325    2300
Name: price, dtype: int64

,bathrooms,bedrooms,square_feet,latitude,longitude,amenities_count,category_housing/rent/home,has_photo_Thumbnail,has_photo_Yes,cityname_2,...,state_40,state_41,state_42,state_43,state_44,state_45,state_46,state_47,state_48,state_49
7095,8.839199,1.46694,2.995746,0.575552,1.378245,1.567179,-0.019391,-3.212674,-0.282009,-0.01371,...,-0.049489,-0.075299,-0.099343,-0.562519,-0.087031,-0.1473,-0.033599,-0.235679,-0.169732,-0.01371
7096,-0.123808,1.46694,2.995746,-0.617837,-1.509388,-0.406072,-0.019391,0.311267,-0.282009,-0.01371,...,-0.049489,-0.075299,-0.099343,-0.562519,-0.087031,-0.1473,-0.033599,-0.235679,-0.169732,-0.01371
7097,8.839199,1.46694,2.995746,0.755014,0.508054,-0.687965,-0.019391,0.311267,-0.282009,-0.01371,...,-0.049489,-0.075299,-0.099343,-0.562519,-0.087031,-0.1473,-0.033599,-0.235679,-0.169732,-0.01371
7098,-0.123808,1.46694,2.995746,0.276989,1.222646,1.003393,-0.019391,0.311267,-0.282009,-0.01371,...,-0.049489,-0.075299,-0.099343,-0.562519,-0.087031,-0.1473,-0.033599,-0.235679,-0.169732,-0.01371
7099,8.839199,1.46694,3.002565,-0.300880,1.103061,2.412858,-0.019391,0.311267,-0.282009,-0.01371,...,-0.049489,-0.075299,-0.099343,-0.562519,-0.087031,-0.1473,-0.033599,-0.235679,-0.169732,-0.01371


7095    2430
7096    2200
7097    1950
7098    1705
7099    1240
Name: price, dtype: int64

Debido a la naturaleza de los datos elegiremos modelos que se ajustan bien a este tipo de problemas:
- Logistic Regression
- SVC
- XGBoost

A continuación realizaremos la **optimización de hiperparámetros** y el **entrenamiento de tres modelos de clasificación binaria**. Para cada modelo:

1. Se utiliza **Optuna** para explorar diferentes combinaciones de hiperparámetros y maximizar la `F1-score` (Esta es la métrica más balanceada ya que es un promedio). Cada combinación de parámetros se evalúa mediante una función objetivo (`objective`) que entrena el modelo, realiza predicciones sobre el conjunto de prueba y calcula métricas de rendimiento como `accuracy`, `precision`, `f1` y `recall`.

2. Se emplea **MLflow** para hacer un seguimiento automático de los experimentos (`autolog`) y registrar los parámetros, métricas y modelos entrenados.

3. Para Logistic Regression y SVC, se crean estudios de Optuna que prueban un número definido de configuraciones (`n_trials=3`) y se seleccionan los mejores parámetros encontrados. Para XGBoost, además se ajustan hiperparámetros como número de árboles, profundidad máxima, tasa de aprendizaje y gamma.

In [10]:
def hp_tuning_rf_reg(X_train_scaled, X_test_scaled, y_train, y_test, X_val=None, y_val=None, n_trials=5):
    def objective_rf(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 1200),
            "max_depth": trial.suggest_int("max_depth", 3, 30),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
            "random_state": 42,
            "n_jobs": -1
        }

        with mlflow.start_run(nested=True):
            # choose evaluation set: prefer validation if provided
            if X_val is not None and y_val is not None:
                eval_X = X_val_scaled
                eval_y = y_val
            else:
                eval_X = X_test_scaled
                eval_y = y_test

            mlflow.set_tag("model_family", "random_forest_regressor")
            mlflow.log_params(params)
            mlflow.log_artifact("../artifacts/preprocessor/encoder.pkl", artifact_path="preprocessor")
            mlflow.log_artifact("../artifacts/preprocessor/scaler.pkl", artifact_path="preprocessor")
            mlflow.log_artifact("../artifacts/preprocessor/feature_columns.pkl", artifact_path="preprocessor")
            
            model = RandomForestRegressor(**params)
            model.fit(X_train_scaled, y_train)

            y_pred = model.predict(eval_X)

            rms = float(np.sqrt(mean_squared_error(eval_y, y_pred)))
            mae = float(mean_absolute_error(eval_y, y_pred))
            r2 = float(r2_score(eval_y, y_pred))

            mlflow.log_metric("rmse", rms)
            mlflow.log_metric("mae", mae)
            mlflow.log_metric("r2", r2)

            signature = infer_signature(eval_X, y_pred)
            mlflow.sklearn.log_model(model, artifact_path="rf_regressor", signature=signature)

        return rms  # Optuna will minimize RMSE

    sampler = TPESampler(seed=42)
    study = optuna.create_study(direction="minimize", sampler=sampler)
    with mlflow.start_run(run_name="RF Regression (Optuna)", nested=True):
        study.optimize(objective_rf, n_trials=n_trials)

    return study.best_params


In [11]:
def hp_tuning_xgb_reg(X_train_scaled, X_test_scaled, y_train, y_test, X_val=None, y_val=None, n_trials=5):
    def objective_xgb(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 300, 2000),
            "max_depth": trial.suggest_int("max_depth", 3, 20),
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "random_state": 42,
            "n_jobs": -1
        }

        with mlflow.start_run(nested=True):
            if X_val is not None and y_val is not None:
                eval_X = X_val_scaled
                eval_y = y_val
            else:
                eval_X = X_test_scaled
                eval_y = y_test

            mlflow.set_tag("model_family", "xgboost_regressor")
            mlflow.log_params(params)
            
            mlflow.log_artifact("../artifacts/preprocessor/encoder.pkl", artifact_path="preprocessor")
            mlflow.log_artifact("../artifacts/preprocessor/scaler.pkl", artifact_path="preprocessor")
            mlflow.log_artifact("../artifacts/preprocessor/feature_columns.pkl", artifact_path="preprocessor")

            model = xgb.XGBRegressor(**params)
            model.fit(X_train_scaled, y_train)

            y_pred = model.predict(eval_X)

            rms = float(np.sqrt(mean_squared_error(eval_y, y_pred)))
            mae = float(mean_absolute_error(eval_y, y_pred))
            r2 = float(r2_score(eval_y, y_pred))

            mlflow.log_metric("rmse", rms)
            mlflow.log_metric("mae", mae)
            mlflow.log_metric("r2", r2)

            signature = infer_signature(eval_X, y_pred)
            mlflow.xgboost.log_model(model, artifact_path="xgb_regressor", signature=signature)

        return rms

    sampler = TPESampler(seed=42)
    study = optuna.create_study(direction="minimize", sampler=sampler)
    with mlflow.start_run(run_name="XGB Regression (Optuna)", nested=True):
        study.optimize(objective_xgb, n_trials=n_trials)

    return study.best_params


In [12]:
def hp_tuning_lgbm_reg(X_train_scaled, X_test_scaled, y_train, y_test, X_val=None, y_val=None, n_trials=5):
    def objective_lgbm(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 200, 2000),
            "max_depth": trial.suggest_int("max_depth", -1, 20),
            "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 16, 256),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "random_state": 42,
            "n_jobs": -1
        }

        with mlflow.start_run(nested=True):
            if X_val is not None and y_val is not None:
                eval_X = X_val_scaled
                eval_y = y_val
            else:
                eval_X = X_test_scaled
                eval_y = y_test

            mlflow.set_tag("model_family", "lightgbm_regressor")
            mlflow.log_params(params)
            
            mlflow.log_artifact("../artifacts/preprocessor/encoder.pkl", artifact_path="preprocessor")
            mlflow.log_artifact("../artifacts/preprocessor/scaler.pkl", artifact_path="preprocessor")
            mlflow.log_artifact("../artifacts/preprocessor/feature_columns.pkl", artifact_path="preprocessor")

            model = lgb.LGBMRegressor(**params)
            model.fit(X_train_scaled, y_train)

            y_pred = model.predict(eval_X)

            rms = float(np.sqrt(mean_squared_error(eval_y, y_pred)))
            mae = float(mean_absolute_error(eval_y, y_pred))
            r2 = float(r2_score(eval_y, y_pred))

            mlflow.log_metric("rmse", rms)
            mlflow.log_metric("mae", mae)
            mlflow.log_metric("r2", r2)

            signature = infer_signature(eval_X, y_pred)
            mlflow.lightgbm.log_model(model, artifact_path="lgbm_regressor", signature=signature)

        return rms

    sampler = TPESampler(seed=42)
    study = optuna.create_study(direction="minimize", sampler=sampler)
    with mlflow.start_run(run_name="LightGBM Regression (Optuna)", nested=True):
        study.optimize(objective_lgbm, n_trials=n_trials)

    return study.best_params

In [13]:
best_rf = hp_tuning_rf_reg(X_train_scaled, X_test_scaled, y_train, y_test, X_val=X_val, y_val=y_val, n_trials=2)
best_xgb = hp_tuning_xgb_reg(X_train_scaled, X_test_scaled, y_train, y_test, X_val=X_val, y_val=y_val, n_trials=2)
best_lgbm = hp_tuning_lgbm_reg(X_train_scaled, X_test_scaled, y_train, y_test, X_val=X_val, y_val=y_val, n_trials=2)

[I 2025-12-02 14:31:33,069] A new study created in memory with name: no-name-e6f0c50e-3097-4e8c-a49b-641fb6c620fe
2025/12/02 14:31:37 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2025/12/02 14:33:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run bittersweet-crab-825 at: https://dbc-2a5806a7-a130.cloud.databricks.com/ml/experiments/1406487778709013/runs/0eaaf66796a6499b93c70540564b4f3b
🧪 View experiment at: https://dbc-2a5806a7-a130.cloud.databricks.com/ml/experiments/1406487778709013


[I 2025-12-02 14:34:56,020] Trial 0 finished with value: 308.45792873785166 and parameters: {'n_estimators': 574, 'max_depth': 29, 'min_samples_split': 15, 'min_samples_leaf': 6}. Best is trial 0 with value: 308.45792873785166.
2025/12/02 14:35:00 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2025/12/02 14:36:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run flawless-chimp-85 at: https://dbc-2a5806a7-a130.cloud.databricks.com/ml/experiments/1406487778709013/runs/a97953820f774c8c8cd6ac357721852f
🧪 View experiment at: https://dbc-2a5806a7-a130.cloud.databricks.com/ml/experiments/1406487778709013


[I 2025-12-02 14:37:59,683] Trial 1 finished with value: 352.49071548918454 and parameters: {'n_estimators': 356, 'max_depth': 7, 'min_samples_split': 3, 'min_samples_leaf': 9}. Best is trial 0 with value: 308.45792873785166.


...

🏃 View run powerful-tern-808 at: https://dbc-2a5806a7-a130.cloud.databricks.com/ml/experiments/1406487778709013/runs/d3871e65457e41729aa14f89b2ae077f
🧪 View experiment at: https://dbc-2a5806a7-a130.cloud.databricks.com/ml/experiments/1406487778709013


[I 2025-12-02 14:44:47,536] Trial 0 finished with value: 300.8091988506982 and parameters: {'n_estimators': 874, 'max_depth': 19, 'learning_rate': 0.06504856968981275, 'num_leaves': 160, 'subsample': 0.5780093202212182, 'colsample_bytree': 0.5779972601681014}. Best is trial 0 with value: 300.8091988506982.


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003007 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1046
[LightGBM] [Info] Number of data points in the train set: 5321, number of used features: 91
[LightGBM] [Info] Start training from score 1171.384326
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gai

2025/12/02 14:44:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
[I 2025-12-02 14:45:18,854] Trial 1 finished with value: 301.70296986534066 and parameters: {'n_estimators': 304, 'max_depth': 18, 'learning_rate': 0.030834348179355788, 'num_leaves': 186, 'subsample': 0.5102922471479012, 'colsample_bytree': 0.9849549260809971}. Best is trial 0 with value: 300.8091988506982.


🏃 View run merciful-owl-330 at: https://dbc-2a5806a7-a130.cloud.databricks.com/ml/experiments/1406487778709013/runs/7b6bbc716bc14feaa3732442808a4165
🧪 View experiment at: https://dbc-2a5806a7-a130.cloud.databricks.com/ml/experiments/1406487778709013
🏃 View run LightGBM Regression (Optuna) at: https://dbc-2a5806a7-a130.cloud.databricks.com/ml/experiments/1406487778709013/runs/09c0e06141dc49d893c9470ee4713b67
🧪 View experiment at: https://dbc-2a5806a7-a130.cloud.databricks.com/ml/experiments/1406487778709013


# MLFLOW Registry
En esta función se entrenan y evalúan los tres modelos que seleccionamos: Logistic Regression, SVC y XGBoost, utilizando los mejores hiperparámetros encontrados previamente. Para cada modelo se registran los parámetros, se calculan métricas de desempeño como accuracy, precision, recall y F1-score, y finalmente se almacenan los modelos en MLflow para su seguimiento y futura reutilización. Lo que buscamos es automatizar el entrenamiento, evaluación y registro de los modelos de manera consistente y reproducible.

In [14]:
def train_best_models(
    X_train_scaled, y_train,
    X_test_scaled, y_test,
    best_params_rf,
    best_params_xgb,
    best_params_lgbm
):
    # 0) FILTRADO DE HIPERPARÁMETROS POR MODELO
    RF_VALID = {
        "n_estimators", "max_depth", "min_samples_split",
        "min_samples_leaf", "max_features", "bootstrap",
        "criterion", "random_state"
    }

    XGB_VALID = {
        "n_estimators", "max_depth", "learning_rate",
        "subsample", "colsample_bytree", "gamma",
        "lambda", "alpha"
    }

    LGBM_VALID = {
        "num_leaves", "learning_rate", "n_estimators",
        "min_child_samples", "subsample", "colsample_bytree",
        "reg_lambda", "reg_alpha"
    }

    def filter_params(params, valid):
        return {k: v for k, v in params.items() if k in valid}

    best_params_rf   = filter_params(best_params_rf, RF_VALID)
    best_params_xgb  = filter_params(best_params_xgb, XGB_VALID)
    best_params_lgbm = filter_params(best_params_lgbm, LGBM_VALID)

    print("RF params usados:", best_params_rf)
    print("XGB params usados:", best_params_xgb)
    print("LGBM params usados:", best_params_lgbm)

    # 1) RANDOM FOREST REGRESSOR
    mlflow.end_run()
    with mlflow.start_run(run_name='Best Random Forest Regressor', nested=True):
        
        mlflow.log_artifact("../artifacts/preprocessor/encoder.pkl", artifact_path="preprocessor")
        mlflow.log_artifact("../artifacts/preprocessor/scaler.pkl", artifact_path="preprocessor")
        mlflow.log_artifact("../artifacts/preprocessor/feature_columns.pkl", artifact_path="preprocessor")
        mlflow.log_params(best_params_rf)

        model = RandomForestRegressor(**best_params_rf)
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)

        mlflow.log_metric("rmse", root_mean_squared_error(y_test, y_pred))
        mlflow.log_metric("mae", mean_absolute_error(y_test, y_pred))
        mlflow.log_metric("r2", r2_score(y_test, y_pred))

        signature = infer_signature(X_train_scaled, model.predict(X_train_scaled))
        mlflow.sklearn.log_model(model, "model", signature=signature)

    # 2) XGBOOST REGRESSOR
    mlflow.end_run()
    with mlflow.start_run(run_name='Best XGBoost Regressor', nested=True):
        
        mlflow.log_artifact("../artifacts/preprocessor/encoder.pkl", artifact_path="preprocessor")
        mlflow.log_artifact("../artifacts/preprocessor/scaler.pkl", artifact_path="preprocessor")
        mlflow.log_artifact("../artifacts/preprocessor/feature_columns.pkl", artifact_path="preprocessor")

        mlflow.log_params(best_params_xgb)

        model = xgb.XGBRegressor(objective='reg:squarederror', **best_params_xgb)
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)

        mlflow.log_metric("rmse", root_mean_squared_error(y_test, y_pred))
        mlflow.log_metric("mae", mean_absolute_error(y_test, y_pred))
        mlflow.log_metric("r2", r2_score(y_test, y_pred))

        signature = infer_signature(X_train_scaled, model.predict(X_train_scaled))
        mlflow.xgboost.log_model(model, "model", signature=signature)

    # 3) LIGHTGBM REGRESSOR
    mlflow.end_run()
    with mlflow.start_run(run_name='Best LightGBM Regressor', nested=True):
        
        mlflow.log_artifact("../artifacts/preprocessor/encoder.pkl", artifact_path="preprocessor")
        mlflow.log_artifact("../artifacts/preprocessor/scaler.pkl", artifact_path="preprocessor")
        mlflow.log_artifact("../artifacts/preprocessor/feature_columns.pkl", artifact_path="preprocessor")

        mlflow.log_params(best_params_lgbm)

        model = lgb.LGBMRegressor(**best_params_lgbm)
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)

        mlflow.log_metric("rmse", root_mean_squared_error(y_test, y_pred))
        mlflow.log_metric("mae", mean_absolute_error(y_test, y_pred))
        mlflow.log_metric("r2", r2_score(y_test, y_pred))

        signature = infer_signature(X_train_scaled, model.predict(X_train_scaled))
        mlflow.lightgbm.log_model(model, "model", signature=signature)


In [15]:
train_best_models(X_train_scaled, y_train, X_test_scaled, y_test, best_lgbm, best_xgb, best_rf)

RF params usados: {'n_estimators': 874, 'max_depth': 19}
XGB params usados: {'n_estimators': 565, 'max_depth': 4, 'learning_rate': 0.13983740016490973, 'subsample': 0.8005575058716043, 'colsample_bytree': 0.8540362888980227}
LGBM params usados: {'n_estimators': 574}


2025/12/02 14:46:23 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2025/12/02 14:53:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Best Random Forest Regressor at: https://dbc-2a5806a7-a130.cloud.databricks.com/ml/experiments/1406487778709013/runs/8e21179503454457a6f7b7ce2f3c1476
🧪 View experiment at: https://dbc-2a5806a7-a130.cloud.databricks.com/ml/experiments/1406487778709013


2025/12/02 14:55:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Best XGBoost Regressor at: https://dbc-2a5806a7-a130.cloud.databricks.com/ml/experiments/1406487778709013/runs/a888d952af5f474182f39dbcda2b635e
🧪 View experiment at: https://dbc-2a5806a7-a130.cloud.databricks.com/ml/experiments/1406487778709013
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008908 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1046
[LightGBM] [Info] Number of data points in the train set: 5321, number of used features: 91
[LightGBM] [Info] Start training from score 1171.384326


2025/12/02 14:55:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Best LightGBM Regressor at: https://dbc-2a5806a7-a130.cloud.databricks.com/ml/experiments/1406487778709013/runs/0cc02dd438844a76823d054855100798
🧪 View experiment at: https://dbc-2a5806a7-a130.cloud.databricks.com/ml/experiments/1406487778709013


Esta función se encarga de registrar automáticamente los dos mejores modelos de un experimento en el **Model Registry** de MLflow y asignarles los alias ya sea como `Champion` y `Challenger`.
1. Primero busca todos los runs marcados como candidatos (`candidate=true`) y los ordena según la métrica F1.
2. Luego selecciona los dos primeros: el de mayor F1 se registra como `Champion` y el segundo como `Challenger`.
3. Cada modelo se registra en el model registry y se le asigna su alias correspondiente.

In [20]:
runs = mlflow.search_runs(
        experiment_names=["/Users/aclarapao@gmail.com/proyecto_final_precios_4"],
        filter_string="tags.candidate = 'true'",
        order_by=[f"metrics.r2"]
    )
runs

,run_id,experiment_id,status,artifact_uri,start_time,end_time


In [22]:
model_name = "workspace.default.proyecto_final_precios_4" # Nombre del modelo registrado en MLflow

client = MlflowClient() 

In [26]:
# Registrar el mejor modelo
result = mlflow.register_model(
    model_uri=f"runs:/a888d952af5f474182f39dbcda2b635e/model", # el id del mejor modelo obtenido
    name=model_name
)

model_version = result.version
new_alias = "Champion"

client.set_registered_model_alias(
    name=model_name,
    alias=new_alias,
    version=result.version
)

Registered model 'workspace.default.proyecto_final_precios_4' already exists. Creating a new version of this model...
2025/12/02 15:05:18 WARNING mlflow.tracking._model_registry.fluent: Run with id a888d952af5f474182f39dbcda2b635e has no artifacts at artifact path 'model', registering model based on models:/m-2004a18b8cc8461f8ed3d61634265c36 instead


Uploading artifacts:   0%|          | 0/6 [00:00<?, ?it/s]

Created version '2' of model 'workspace.default.proyecto_final_precios_4'.


In [ ]:
# Registrar el 2do mejor modelo
result = mlflow.register_model(
    model_uri=f"runs:/0cc02dd438844a76823d054855100798/model", # el id del mejor modelo obtenido
    name=model_name
)

model_version = result.version
new_alias = "Champion"

client.set_registered_model_alias(
    name=model_name,
    alias=new_alias,
    version=result.version
)

In [27]:
# Registrar el mejor modelo
result = mlflow.register_model(
    model_uri=f"runs:/0cc02dd438844a76823d054855100798/model", # id de uno de los mejores modelos obtenidos, xgboost
    name=model_name
)

model_version = result.version
new_alias = "Challenger"

client.set_registered_model_alias(
    name=model_name,
    alias=new_alias,
    version=result.version
)

Registered model 'workspace.default.proyecto_final_precios_4' already exists. Creating a new version of this model...
2025/12/02 15:05:38 WARNING mlflow.tracking._model_registry.fluent: Run with id 0cc02dd438844a76823d054855100798 has no artifacts at artifact path 'model', registering model based on models:/m-ce7bbfc13dfc4a04bd750e0cc823af10 instead


Uploading artifacts:   0%|          | 0/6 [00:00<?, ?it/s]

Created version '3' of model 'workspace.default.proyecto_final_precios_4'.


Esta función se encarga de registrar automáticamente los dos mejores modelos de un experimento en el **Model Registry** de MLflow y asignarles los alias ya sea como `Champion` y `Challenger`.
1. Primero busca todos los runs marcados como candidatos (`candidate=true`) y los ordena según la métrica F1.
2. Luego selecciona los dos primeros: el de mayor F1 se registra como `Champion` y el segundo como `Challenger`.
3. Cada modelo se registra en el model registry y se le asigna su alias correspondiente.